In [1]:
# pip install pyserial

import time
import tkinter as tk
from tkinter import ttk, messagebox
import serial
import serial.tools.list_ports


class BT1001L:
    def __init__(self, port, addr=31, timeout=1.0):
        self.port = port
        self.addr = int(addr)
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data):
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send(self, pdu, wait_s=0.25):
        if self.ser is None or not self.ser.is_open:
            self.open()

        frame = self._frame(pdu)
        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        resp = self.ser.read_all()

        return frame, resp

    def set_speed(self, rpm, start=True, cw=True):
        rpm = max(0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))
        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self):
        return self.set_speed(0.0, start=False)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.6)


class PumpPanel(ttk.LabelFrame):
    def __init__(self, master, title):
        super().__init__(master, text=title, padding=10)

        self.pump = None

        self.port_var = tk.StringVar()
        self.addr_var = tk.StringVar(value="31")
        self.rpm_var = tk.StringVar(value="20.0")
        self.dir_var = tk.StringVar(value="CW")
        self.status_var = tk.StringVar(value="Disconnected")
        self.last_tx_var = tk.StringVar(value="-")
        self.last_rx_var = tk.StringVar(value="-")

        ttk.Label(self, text="COM Port").grid(row=0, column=0, sticky="w")
        self.port_box = ttk.Combobox(self, textvariable=self.port_var, width=14)
        self.port_box.grid(row=0, column=1, sticky="ew", padx=5)

        ttk.Button(self, text="Refresh", command=self.refresh_ports).grid(row=0, column=2)

        ttk.Label(self, text="Address").grid(row=1, column=0, sticky="w")
        ttk.Entry(self, textvariable=self.addr_var, width=10).grid(row=1, column=1, sticky="w", padx=5)

        ttk.Label(self, text="RPM").grid(row=2, column=0, sticky="w")
        ttk.Entry(self, textvariable=self.rpm_var, width=10).grid(row=2, column=1, sticky="w", padx=5)

        ttk.Label(self, text="Direction").grid(row=3, column=0, sticky="w")
        ttk.Combobox(self, textvariable=self.dir_var, values=["CW", "CCW"], width=8, state="readonly").grid(
            row=3, column=1, sticky="w", padx=5
        )

        ttk.Button(self, text="Connect", command=self.connect).grid(row=4, column=0, pady=8, sticky="ew")
        ttk.Button(self, text="Disconnect", command=self.disconnect).grid(row=4, column=1, pady=8, sticky="ew")
        ttk.Button(self, text="Start", command=self.start).grid(row=5, column=0, pady=4, sticky="ew")
        ttk.Button(self, text="Stop", command=self.stop).grid(row=5, column=1, pady=4, sticky="ew")
        ttk.Button(self, text="Read Speed", command=self.read_speed).grid(row=5, column=2, pady=4, sticky="ew")

        ttk.Label(self, text="Status").grid(row=6, column=0, sticky="w")
        ttk.Label(self, textvariable=self.status_var).grid(row=6, column=1, columnspan=2, sticky="w")

        ttk.Label(self, text="Last TX").grid(row=7, column=0, sticky="w")
        ttk.Label(self, textvariable=self.last_tx_var, wraplength=360).grid(row=7, column=1, columnspan=2, sticky="w")

        ttk.Label(self, text="Last RX").grid(row=8, column=0, sticky="w")
        ttk.Label(self, textvariable=self.last_rx_var, wraplength=360).grid(row=8, column=1, columnspan=2, sticky="w")

        self.columnconfigure(1, weight=1)
        self.refresh_ports()

    def refresh_ports(self):
        ports = [p.device for p in serial.tools.list_ports.comports()]
        self.port_box["values"] = ports
        if ports and not self.port_var.get():
            self.port_var.set(ports[0])

    def connect(self):
        try:
            self.pump = BT1001L(
                port=self.port_var.get(),
                addr=int(self.addr_var.get()),
            )
            self.pump.open()
            self.status_var.set("Connected")
        except Exception as e:
            self.status_var.set("Connection failed")
            messagebox.showerror("Connection error", str(e))

    def disconnect(self):
        try:
            if self.pump:
                self.pump.close()
            self.status_var.set("Disconnected")
        except Exception as e:
            messagebox.showerror("Disconnect error", str(e))

    def _ensure_connected(self):
        if self.pump is None:
            self.connect()
        if self.pump is None:
            raise RuntimeError("Pump not connected")

    def _show_io(self, frame, resp):
        self.last_tx_var.set(frame.hex(" "))
        self.last_rx_var.set(resp.hex(" ") if resp else "<no response>")

    def start(self):
        try:
            self._ensure_connected()
            rpm = float(self.rpm_var.get())
            cw = self.dir_var.get() == "CW"
            frame, resp = self.pump.set_speed(rpm, start=True, cw=cw)
            self._show_io(frame, resp)
            self.status_var.set(f"Running {rpm} rpm {self.dir_var.get()}")
        except Exception as e:
            messagebox.showerror("Start error", str(e))

    def stop(self):
        try:
            self._ensure_connected()
            frame, resp = self.pump.stop()
            self._show_io(frame, resp)
            self.status_var.set("Stopped")
        except Exception as e:
            messagebox.showerror("Stop error", str(e))

    def read_speed(self):
        try:
            self._ensure_connected()
            frame, resp = self.pump.read_speed()
            self._show_io(frame, resp)
            if int(self.addr_var.get()) == 31:
                self.status_var.set("Broadcast address: no RX expected")
            else:
                self.status_var.set("Read command sent")
        except Exception as e:
            messagebox.showerror("Read error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop()
        except Exception:
            pass


class App(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("BT100-1L Dual Pump Controller")
        self.geometry("900x560")

        self.pump1 = PumpPanel(self, "Pump 1")
        self.pump1.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")

        self.pump2 = PumpPanel(self, "Pump 2")
        self.pump2.grid(row=0, column=1, padx=10, pady=10, sticky="nsew")

        control = ttk.Frame(self, padding=10)
        control.grid(row=1, column=0, columnspan=2, sticky="ew")

        ttk.Button(control, text="START BOTH", command=self.start_both).pack(side="left", padx=5)
        ttk.Button(control, text="STOP BOTH", command=self.stop_both).pack(side="left", padx=5)
        ttk.Button(control, text="EMERGENCY STOP", command=self.emergency_stop).pack(side="right", padx=5)

        note = (
            "Tip: addr=31 is broadcast. It controls all pumps but gives no RX response. "
            "For independent control on one RS485 bus, set pump IDs to different addresses."
        )
        ttk.Label(self, text=note, wraplength=860).grid(row=2, column=0, columnspan=2, padx=10, pady=5)

        self.columnconfigure(0, weight=1)
        self.columnconfigure(1, weight=1)
        self.rowconfigure(0, weight=1)

        self.protocol("WM_DELETE_WINDOW", self.on_close)

    def start_both(self):
        self.pump1.start()
        self.pump2.start()

    def stop_both(self):
        self.pump1.stop()
        self.pump2.stop()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        messagebox.showinfo("Emergency stop", "Stop commands sent.")

    def on_close(self):
        self.emergency_stop()
        self.pump1.disconnect()
        self.pump2.disconnect()
        self.destroy()


if __name__ == "__main__":
    app = App()
    app.mainloop()

In [2]:
# pip install pyqt6 pyserial

import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox, QLineEdit,
    QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout, QGroupBox,
    QMessageBox
)
from PyQt6.QtCore import Qt


class BT1001L:
    def __init__(self, port="COM10", addr=31, timeout=1.0):
        self.port = port
        self.addr = int(addr)
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu: bytes) -> int:
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _frame(self, pdu: bytes) -> bytes:
        fcs = self._checksum(pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send(self, pdu: bytes, wait_s=0.25):
        self.open()

        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        resp = self.ser.read_all()

        return frame, resp

    def set_speed(self, rpm, start=True, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self):
        return self.set_speed(0.0, start=False)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.6)


class PumpPanel(QGroupBox):
    def __init__(self, title):
        super().__init__(title)

        self.pump = None

        self.port_box = QComboBox()
        self.port_box.setEditable(True)

        self.addr_edit = QLineEdit("31")
        self.rpm_edit = QLineEdit("20.0")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.status_label = QLabel("Disconnected")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(140)

        self.refresh_btn = QPushButton("Refresh Ports")
        self.connect_btn = QPushButton("Connect")
        self.disconnect_btn = QPushButton("Disconnect")
        self.start_btn = QPushButton("Start")
        self.stop_btn = QPushButton("Stop")
        self.read_btn = QPushButton("Read Speed")

        layout = QGridLayout()

        layout.addWidget(QLabel("COM Port"), 0, 0)
        layout.addWidget(self.port_box, 0, 1)
        layout.addWidget(self.refresh_btn, 0, 2)

        layout.addWidget(QLabel("Address"), 1, 0)
        layout.addWidget(self.addr_edit, 1, 1)

        layout.addWidget(QLabel("RPM"), 2, 0)
        layout.addWidget(self.rpm_edit, 2, 1)

        layout.addWidget(QLabel("Direction"), 3, 0)
        layout.addWidget(self.dir_box, 3, 1)

        layout.addWidget(self.connect_btn, 4, 0)
        layout.addWidget(self.disconnect_btn, 4, 1)

        layout.addWidget(self.start_btn, 5, 0)
        layout.addWidget(self.stop_btn, 5, 1)
        layout.addWidget(self.read_btn, 5, 2)

        layout.addWidget(QLabel("Status"), 6, 0)
        layout.addWidget(self.status_label, 6, 1, 1, 2)

        layout.addWidget(self.log, 7, 0, 1, 3)

        self.setLayout(layout)

        self.refresh_btn.clicked.connect(self.refresh_ports)
        self.connect_btn.clicked.connect(self.connect_pump)
        self.disconnect_btn.clicked.connect(self.disconnect_pump)
        self.start_btn.clicked.connect(self.start_pump)
        self.stop_btn.clicked.connect(self.stop_pump)
        self.read_btn.clicked.connect(self.read_speed)

        self.refresh_ports()

    def refresh_ports(self):
        current = self.port_box.currentText().strip()

        self.port_box.clear()

        ports = [p.device for p in serial.tools.list_ports.comports()]
        self.port_box.addItems(ports)

        # Add common COM range manually so COMxx is easy to pick/type
        for i in range(1, 257):
            name = f"COM{i}"
            if name not in ports:
                self.port_box.addItem(name)

        if current:
            self.port_box.setCurrentText(current)
        elif "COM10" in [self.port_box.itemText(i) for i in range(self.port_box.count())]:
            self.port_box.setCurrentText("COM10")

    def log_io(self, frame=None, resp=None, msg=None):
        if msg:
            self.log.append(msg)

        if frame is not None:
            self.log.append(f"TX: {frame.hex(' ')}")

        if resp is not None:
            rx = resp.hex(" ") if resp else "<no response>"
            self.log.append(f"RX: {rx}")

        self.log.append("")

    def get_port(self):
        port = self.port_box.currentText().strip().upper()
        if not port:
            raise ValueError("COM port is empty")
        return port

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be 1 to 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be 0 to 100")
        return rpm

    def connect_pump(self):
        try:
            self.disconnect_pump(show_msg=False)

            self.pump = BT1001L(
                port=self.get_port(),
                addr=self.get_addr(),
            )
            self.pump.open()

            self.status_label.setText("Connected")
            self.log_io(msg=f"Connected to {self.get_port()} addr={self.get_addr()}")

        except Exception as e:
            self.status_label.setText("Connection failed")
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_msg=True):
        try:
            if self.pump:
                self.pump.close()
                self.pump = None

            self.status_label.setText("Disconnected")

            if show_msg:
                self.log_io(msg="Disconnected")

        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()
        if self.pump is None:
            raise RuntimeError("Pump is not connected")

        # Update address/port from UI before sending
        self.pump.addr = self.get_addr()
        self.pump.port = self.get_port()

    def start_pump(self):
        try:
            self.ensure_pump()

            rpm = self.get_rpm()
            cw = self.dir_box.currentText() == "CW"

            frame, resp = self.pump.set_speed(rpm, start=True, cw=cw)

            self.status_label.setText(f"Running {rpm} rpm {self.dir_box.currentText()}")
            self.log_io(frame, resp)

            if self.get_addr() == 31:
                self.log_io(msg="Broadcast address 31: no RX expected.")

        except Exception as e:
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            self.ensure_pump()

            frame, resp = self.pump.stop()

            self.status_label.setText("Stopped")
            self.log_io(frame, resp)

            if self.get_addr() == 31:
                self.log_io(msg="Broadcast address 31: no RX expected.")

        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_speed(self):
        try:
            self.ensure_pump()

            frame, resp = self.pump.read_speed()

            self.status_label.setText("Read sent")
            self.log_io(frame, resp)

            if self.get_addr() == 31:
                self.log_io(msg="Broadcast address 31 cannot read response.")

        except Exception as e:
            QMessageBox.critical(self, "Read Error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop()
        except Exception:
            pass


class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("BT100-1L Dual Pump Controller - PyQt6")
        self.resize(1100, 650)

        self.pump1 = PumpPanel("Pump 1")
        self.pump2 = PumpPanel("Pump 2")

        self.start_both_btn = QPushButton("START BOTH")
        self.stop_both_btn = QPushButton("STOP BOTH")
        self.emergency_btn = QPushButton("EMERGENCY STOP")

        self.emergency_btn.setStyleSheet(
            "QPushButton { background-color: #b00020; color: white; font-weight: bold; padding: 8px; }"
        )

        top_layout = QHBoxLayout()
        top_layout.addWidget(self.pump1)
        top_layout.addWidget(self.pump2)

        bottom_layout = QHBoxLayout()
        bottom_layout.addWidget(self.start_both_btn)
        bottom_layout.addWidget(self.stop_both_btn)
        bottom_layout.addStretch()
        bottom_layout.addWidget(self.emergency_btn)

        note = QLabel(
            "Use addr=31 for broadcast control. Broadcast moves all pumps but gives no RX response. "
            "For independent control on one RS485 bus, assign different pump IDs, e.g. 1 and 2."
        )
        note.setWordWrap(True)

        main_layout = QVBoxLayout()
        main_layout.addLayout(top_layout)
        main_layout.addLayout(bottom_layout)
        main_layout.addWidget(note)

        self.setLayout(main_layout)

        self.start_both_btn.clicked.connect(self.start_both)
        self.stop_both_btn.clicked.connect(self.stop_both)
        self.emergency_btn.clicked.connect(self.emergency_stop)

    def start_both(self):
        self.pump1.start_pump()
        self.pump2.start_pump()

    def stop_both(self):
        self.pump1.stop_pump()
        self.pump2.stop_pump()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        self.pump1.disconnect_pump(show_msg=False)
        self.pump2.disconnect_pump(show_msg=False)
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())

SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
import time
import serial


class BT1001L:
    def __init__(self, port="COM10", addr=1, timeout=1.5):
        self.port = port
        self.addr = addr
        self.timeout = timeout
        self.ser = None

    # ---------- connection ----------
    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
            print(f"Opened {self.port}")
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            print(f"Closed {self.port}")

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    # ---------- protocol ----------
    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data):
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _unescape(self, data):
        out = bytearray()
        i = 0
        while i < len(data):
            if data[i] == 0xE8:
                if i + 1 >= len(data):
                    break
                if data[i + 1] == 0x00:
                    out.append(0xE8)
                elif data[i + 1] == 0x01:
                    out.append(0xE9)
                i += 2
            else:
                out.append(data[i])
                i += 1
        return bytes(out)

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send(self, pdu, wait_s=0.5):
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()

        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        raw = self.ser.read_all()

        print("TX:", frame.hex(" "))
        print("RX:", raw.hex(" ") if raw else "<no response>")

        return raw

    # ---------- commands ----------
    def set_speed(self, rpm, start=True, cw=True):
        rpm = max(0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL")

    def stop(self):
        return self.set_speed(0.0, start=False)


# ---------- decoder ----------
def decode_speed(rx: bytes):
    if not rx:
        return "No response"

    if rx[0] != 0xE9:
        return "Invalid frame"

    addr = rx[1]
    cmd = rx[3:5].decode()

    speed_raw = int.from_bytes(rx[5:7], "big")
    rpm = speed_raw / 10.0

    state1 = rx[7]
    state2 = rx[8]

    running = bool(state1 & 0x01)
    prime = bool(state1 & 0x02)
    direction = "CW" if state2 & 0x01 else "CCW"

    return {
        "address": addr,
        "command": cmd,
        "rpm": rpm,
        "running": running,
        "prime": prime,
        "direction": direction
    }


# ---------- example usage ----------
if __name__ == "__main__":

    with BT1001L("COM10", addr=1) as pump:

        print("\n--- SET SPEED ---")
        pump.set_speed(30.0, start=True, cw=True)

        time.sleep(1)

        print("\n--- READ SPEED ---")
        rx = pump.read_speed()

        decoded = decode_speed(rx)
        print("\nDecoded:", decoded)

        time.sleep(2)

        print("\n--- STOP ---")
        pump.stop()

Opened COM10

--- SET SPEED ---
TX: e9 01 06 58 4c 01 2c 01 01 3e
RX: e9 01 02 58 4c 17

--- READ SPEED ---
TX: e9 01 02 44 4c 0b
RX: e9 01 06 44 4c 01 2c 01 01 22

Decoded: {'address': 1, 'command': 'DL', 'rpm': 30.0, 'running': True, 'prime': False, 'direction': 'CW'}

--- STOP ---
TX: e9 01 06 58 4c 00 00 00 01 12
RX: e9 01 02 58 4c 17
Closed COM10


In [1]:
import sys
import time
import csv
from dataclasses import dataclass

import serial
import serial.tools.list_ports

from PyQt6.QtCore import Qt, QTimer, QThread, pyqtSignal
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox, QLineEdit,
    QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout, QGroupBox,
    QMessageBox, QFileDialog, QTableWidget, QTableWidgetItem
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class BT1001L:
    def __init__(self, port="COM10", addr=1, timeout=1.0):
        self.port = port
        self.addr = int(addr)
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu: bytes, addr=None) -> int:
        if addr is None:
            addr = self.addr
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _unescape(self, data: bytes) -> bytes:
        out = bytearray()
        i = 0
        while i < len(data):
            if data[i] == 0xE8:
                if i + 1 >= len(data):
                    raise ValueError("Bad escape sequence")
                if data[i + 1] == 0x00:
                    out.append(0xE8)
                elif data[i + 1] == 0x01:
                    out.append(0xE9)
                else:
                    raise ValueError("Bad escape sequence")
                i += 2
            else:
                out.append(data[i])
                i += 1
        return bytes(out)

    def _frame(self, pdu: bytes) -> bytes:
        fcs = self._checksum(pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send(self, pdu: bytes, wait_s=0.25):
        self.open()

        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        raw = self.ser.read_all()

        return frame, raw

    def set_speed(self, rpm: float, start=True, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.5)

    def stop(self):
        return self.set_speed(0.0, start=False)

    def set_flow(self, flow_ml_min: float, pump_head: int, tube_no: int,
                 start=True, cw=True):
        flow_ml_min = max(0.0, float(flow_ml_min))
        flow_nl_min = int(round(flow_ml_min * 1_000_000))

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = (
            b"WL"
            + flow_nl_min.to_bytes(4, "big")
            + bytes([state1, state2, int(pump_head), int(tube_no)])
        )
        return self.send(pdu)

    def read_flow(self):
        return self.send(b"RL", wait_s=0.5)


def decode_speed(rx: bytes):
    if not rx:
        return None
    if rx[0] != 0xE9:
        return None
    if len(rx) < 10:
        return None

    addr = rx[1]
    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    speed_raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "address": addr,
        "rpm": speed_raw / 10.0,
        "running": bool(state1 & 0x01),
        "prime": bool(state1 & 0x02),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


def decode_flow(rx: bytes):
    if not rx:
        return None
    if rx[0] != 0xE9:
        return None
    if len(rx) < 14:
        return None

    addr = rx[1]
    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 10 or pdu[:2] != b"RL":
        return None

    flow_nl_min = int.from_bytes(pdu[2:6], "big")
    state1 = pdu[6]
    state2 = pdu[7]

    return {
        "address": addr,
        "flow_ml_min": flow_nl_min / 1_000_000,
        "running": bool(state1 & 0x01),
        "prime": bool(state1 & 0x02),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "pump_head": pdu[8],
        "tube_no": pdu[9],
        "raw": rx.hex(" "),
    }


@dataclass
class Step:
    mode: str
    value: float
    duration_s: float
    direction: str


class SequenceWorker(QThread):
    log = pyqtSignal(str)
    finished_ok = pyqtSignal()
    failed = pyqtSignal(str)

    def __init__(self, pump, steps, pump_head, tube_no):
        super().__init__()
        self.pump = pump
        self.steps = steps
        self.pump_head = pump_head
        self.tube_no = tube_no
        self._stop_requested = False

    def stop_requested(self):
        self._stop_requested = True

    def run(self):
        try:
            for i, step in enumerate(self.steps, start=1):
                if self._stop_requested:
                    break

                cw = step.direction == "CW"
                self.log.emit(f"Step {i}: {step.mode} {step.value}, {step.duration_s}s, {step.direction}")

                if step.mode == "Speed":
                    self.pump.set_speed(step.value, start=True, cw=cw)
                else:
                    self.pump.set_flow(step.value, self.pump_head, self.tube_no, start=True, cw=cw)

                t0 = time.time()
                while time.time() - t0 < step.duration_s:
                    if self._stop_requested:
                        break
                    time.sleep(0.1)

                self.pump.stop()

            self.pump.stop()
            self.finished_ok.emit()

        except Exception as e:
            try:
                self.pump.stop()
            except Exception:
                pass
            self.failed.emit(str(e))


class PlotCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(5, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.t = []
        self.rpm = []

        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)

    def add_point(self, t, rpm):
        self.t.append(t)
        self.rpm.append(rpm)

        if len(self.t) > 300:
            self.t = self.t[-300:]
            self.rpm = self.rpm[-300:]

        self.ax.clear()
        self.ax.plot(self.t, self.rpm)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def clear(self):
        self.t.clear()
        self.rpm.clear()
        self.ax.clear()
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()


class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("BT100-1L Pump 1 Controller")
        self.resize(1200, 780)

        self.pump = None
        self.monitor_start_time = None
        self.monitor_data = []
        self.sequence_worker = None

        self.port_box = QComboBox()
        self.port_box.setEditable(True)

        self.addr_edit = QLineEdit("1")
        self.rpm_edit = QLineEdit("30.0")
        self.flow_edit = QLineEdit("3.0")
        self.pump_head_edit = QLineEdit("2")
        self.tube_no_edit = QLineEdit("3")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.monitor_interval_edit = QLineEdit("1.0")

        self.status_label = QLabel("Disconnected")
        self.feedback_label = QLabel("-")

        self.log = QTextEdit()
        self.log.setReadOnly(True)

        self.plot = PlotCanvas()

        self.monitor_timer = QTimer()
        self.monitor_timer.timeout.connect(self.monitor_tick)

        self.sequence_table = QTableWidget(0, 4)
        self.sequence_table.setHorizontalHeaderLabels(["Mode", "Value", "Duration s", "Direction"])

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        conn = QGroupBox("Connection")
        conn_lay = QGridLayout()

        refresh_btn = QPushButton("Refresh Ports")
        connect_btn = QPushButton("Connect")
        disconnect_btn = QPushButton("Disconnect")

        refresh_btn.clicked.connect(self.refresh_ports)
        connect_btn.clicked.connect(self.connect_pump)
        disconnect_btn.clicked.connect(self.disconnect_pump)

        conn_lay.addWidget(QLabel("COM Port"), 0, 0)
        conn_lay.addWidget(self.port_box, 0, 1)
        conn_lay.addWidget(refresh_btn, 0, 2)
        conn_lay.addWidget(QLabel("Address"), 1, 0)
        conn_lay.addWidget(self.addr_edit, 1, 1)
        conn_lay.addWidget(connect_btn, 2, 0)
        conn_lay.addWidget(disconnect_btn, 2, 1)
        conn_lay.addWidget(QLabel("Status"), 3, 0)
        conn_lay.addWidget(self.status_label, 3, 1, 1, 2)
        conn.setLayout(conn_lay)

        control = QGroupBox("Manual Control")
        ctl_lay = QGridLayout()

        set_speed_btn = QPushButton("Set Speed + Start")
        read_speed_btn = QPushButton("Read Speed")
        stop_btn = QPushButton("STOP")
        set_flow_btn = QPushButton("Set Flow + Start")
        read_flow_btn = QPushButton("Read Flow")

        set_speed_btn.clicked.connect(self.set_speed)
        read_speed_btn.clicked.connect(self.read_speed)
        stop_btn.clicked.connect(self.stop_pump)
        set_flow_btn.clicked.connect(self.set_flow)
        read_flow_btn.clicked.connect(self.read_flow)

        ctl_lay.addWidget(QLabel("RPM"), 0, 0)
        ctl_lay.addWidget(self.rpm_edit, 0, 1)
        ctl_lay.addWidget(QLabel("Flow mL/min"), 1, 0)
        ctl_lay.addWidget(self.flow_edit, 1, 1)
        ctl_lay.addWidget(QLabel("Pump head no."), 2, 0)
        ctl_lay.addWidget(self.pump_head_edit, 2, 1)
        ctl_lay.addWidget(QLabel("Tube no."), 3, 0)
        ctl_lay.addWidget(self.tube_no_edit, 3, 1)
        ctl_lay.addWidget(QLabel("Direction"), 4, 0)
        ctl_lay.addWidget(self.dir_box, 4, 1)

        ctl_lay.addWidget(set_speed_btn, 5, 0)
        ctl_lay.addWidget(read_speed_btn, 5, 1)
        ctl_lay.addWidget(set_flow_btn, 6, 0)
        ctl_lay.addWidget(read_flow_btn, 6, 1)
        ctl_lay.addWidget(stop_btn, 7, 0, 1, 2)

        control.setLayout(ctl_lay)

        monitor = QGroupBox("Real-Time Monitoring")
        mon_lay = QGridLayout()

        start_mon_btn = QPushButton("Start Monitor")
        stop_mon_btn = QPushButton("Stop Monitor")
        clear_plot_btn = QPushButton("Clear Plot")
        save_csv_btn = QPushButton("Save CSV")

        start_mon_btn.clicked.connect(self.start_monitor)
        stop_mon_btn.clicked.connect(self.stop_monitor)
        clear_plot_btn.clicked.connect(self.clear_plot)
        save_csv_btn.clicked.connect(self.save_csv)

        mon_lay.addWidget(QLabel("Interval s"), 0, 0)
        mon_lay.addWidget(self.monitor_interval_edit, 0, 1)
        mon_lay.addWidget(start_mon_btn, 1, 0)
        mon_lay.addWidget(stop_mon_btn, 1, 1)
        mon_lay.addWidget(clear_plot_btn, 2, 0)
        mon_lay.addWidget(save_csv_btn, 2, 1)
        mon_lay.addWidget(QLabel("Feedback"), 3, 0)
        mon_lay.addWidget(self.feedback_label, 3, 1)

        monitor.setLayout(mon_lay)

        sequence = QGroupBox("Automation Sequence")
        seq_lay = QVBoxLayout()

        add_speed_btn = QPushButton("Add Speed Step")
        add_flow_btn = QPushButton("Add Flow Step")
        remove_step_btn = QPushButton("Remove Selected")
        run_seq_btn = QPushButton("Run Sequence")
        stop_seq_btn = QPushButton("Stop Sequence")

        add_speed_btn.clicked.connect(lambda: self.add_step("Speed"))
        add_flow_btn.clicked.connect(lambda: self.add_step("Flow"))
        remove_step_btn.clicked.connect(self.remove_selected_step)
        run_seq_btn.clicked.connect(self.run_sequence)
        stop_seq_btn.clicked.connect(self.stop_sequence)

        seq_btn_lay = QHBoxLayout()
        seq_btn_lay.addWidget(add_speed_btn)
        seq_btn_lay.addWidget(add_flow_btn)
        seq_btn_lay.addWidget(remove_step_btn)
        seq_btn_lay.addWidget(run_seq_btn)
        seq_btn_lay.addWidget(stop_seq_btn)

        seq_lay.addWidget(self.sequence_table)
        seq_lay.addLayout(seq_btn_lay)
        sequence.setLayout(seq_lay)

        left = QVBoxLayout()
        left.addWidget(conn)
        left.addWidget(control)
        left.addWidget(monitor)
        left.addStretch()

        right = QVBoxLayout()
        right.addWidget(self.plot)
        right.addWidget(sequence)
        right.addWidget(QLabel("Log"))
        right.addWidget(self.log)

        main = QHBoxLayout()
        main.addLayout(left, 1)
        main.addLayout(right, 2)

        self.setLayout(main)

    def refresh_ports(self):
        current = self.port_box.currentText().strip()
        self.port_box.clear()

        listed = [p.device for p in serial.tools.list_ports.comports()]
        self.port_box.addItems(listed)

        for i in range(1, 257):
            port = f"COM{i}"
            if port not in listed:
                self.port_box.addItem(port)

        if current:
            self.port_box.setCurrentText(current)
        else:
            self.port_box.setCurrentText("COM10")

    def log_msg(self, msg):
        self.log.append(msg)

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def get_port(self):
        return self.port_box.currentText().strip().upper()

    def get_addr(self):
        return int(self.addr_edit.text().strip())

    def get_direction(self):
        return self.dir_box.currentText() == "CW"

    def get_pump_head(self):
        return int(self.pump_head_edit.text().strip())

    def get_tube_no(self):
        return int(self.tube_no_edit.text().strip())

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()
        if self.pump is None:
            raise RuntimeError("Pump not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()
        return self.pump

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)
            self.pump = BT1001L(self.get_port(), self.get_addr())
            self.pump.open()
            self.status_label.setText(f"Connected {self.get_port()} addr={self.get_addr()}")
            self.log_msg(f"Connected {self.get_port()} addr={self.get_addr()}")
        except Exception as e:
            self.pump = None
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        try:
            if self.pump:
                self.pump.close()
                self.pump = None
            self.status_label.setText("Disconnected")
            if show_log:
                self.log_msg("Disconnected")
        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def set_speed(self):
        try:
            pump = self.ensure_pump()
            rpm = float(self.rpm_edit.text().strip())
            frame, rx = pump.set_speed(rpm, start=True, cw=self.get_direction())
            self.log_io(frame, rx)
            self.status_label.setText(f"Running speed {rpm} rpm")
        except Exception as e:
            QMessageBox.critical(self, "Set Speed Error", str(e))

    def read_speed(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            self.log_io(frame, rx)
            decoded = decode_speed(rx)
            if decoded:
                text = (
                    f"{decoded['rpm']} rpm | "
                    f"{'running' if decoded['running'] else 'stopped'} | "
                    f"{decoded['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                self.feedback_label.setText("No valid speed response")
        except Exception as e:
            QMessageBox.critical(self, "Read Speed Error", str(e))

    def set_flow(self):
        try:
            pump = self.ensure_pump()
            flow = float(self.flow_edit.text().strip())
            frame, rx = pump.set_flow(
                flow,
                pump_head=self.get_pump_head(),
                tube_no=self.get_tube_no(),
                start=True,
                cw=self.get_direction(),
            )
            self.log_io(frame, rx)
            self.status_label.setText(f"Running flow {flow} mL/min")
        except Exception as e:
            QMessageBox.critical(self, "Set Flow Error", str(e))

    def read_flow(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_flow()
            self.log_io(frame, rx)
            decoded = decode_flow(rx)
            if decoded:
                text = (
                    f"{decoded['flow_ml_min']} mL/min | "
                    f"{'running' if decoded['running'] else 'stopped'} | "
                    f"{decoded['direction']} | "
                    f"head {decoded['pump_head']} tube {decoded['tube_no']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                self.feedback_label.setText("No valid flow response")
        except Exception as e:
            QMessageBox.critical(self, "Read Flow Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.stop()
            self.log_io(frame, rx)
            self.status_label.setText("Stopped")
        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def start_monitor(self):
        try:
            interval_s = float(self.monitor_interval_edit.text().strip())
            interval_ms = max(200, int(interval_s * 1000))
            self.monitor_start_time = time.time()
            self.monitor_timer.start(interval_ms)
            self.log_msg("Monitoring started")
        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        self.monitor_timer.stop()
        self.log_msg("Monitoring stopped")

    def monitor_tick(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            decoded = decode_speed(rx)

            if decoded:
                t = time.time() - self.monitor_start_time
                rpm = decoded["rpm"]
                self.plot.add_point(t, rpm)

                row = {
                    "time_s": t,
                    "rpm": rpm,
                    "running": decoded["running"],
                    "direction": decoded["direction"],
                    "raw": decoded["raw"],
                }
                self.monitor_data.append(row)

                self.feedback_label.setText(
                    f"{rpm} rpm | {'running' if decoded['running'] else 'stopped'} | {decoded['direction']}"
                )
            else:
                self.feedback_label.setText("No valid speed response")

        except Exception as e:
            self.monitor_timer.stop()
            QMessageBox.critical(self, "Monitor Error", str(e))

    def clear_plot(self):
        self.plot.clear()
        self.monitor_data.clear()

    def save_csv(self):
        if not self.monitor_data:
            QMessageBox.information(self, "Save CSV", "No monitor data to save.")
            return

        path, _ = QFileDialog.getSaveFileName(self, "Save CSV", "bt100_monitor.csv", "CSV files (*.csv)")
        if not path:
            return

        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["time_s", "rpm", "running", "direction", "raw"])
            writer.writeheader()
            writer.writerows(self.monitor_data)

        self.log_msg(f"Saved CSV: {path}")

    def add_step(self, mode):
        row = self.sequence_table.rowCount()
        self.sequence_table.insertRow(row)

        value = self.rpm_edit.text().strip() if mode == "Speed" else self.flow_edit.text().strip()
        duration = "5"
        direction = self.dir_box.currentText()

        for col, item in enumerate([mode, value, duration, direction]):
            self.sequence_table.setItem(row, col, QTableWidgetItem(item))

    def remove_selected_step(self):
        row = self.sequence_table.currentRow()
        if row >= 0:
            self.sequence_table.removeRow(row)

    def get_sequence_steps(self):
        steps = []
        for row in range(self.sequence_table.rowCount()):
            mode = self.sequence_table.item(row, 0).text().strip()
            value = float(self.sequence_table.item(row, 1).text().strip())
            duration_s = float(self.sequence_table.item(row, 2).text().strip())
            direction = self.sequence_table.item(row, 3).text().strip().upper()

            if mode not in ["Speed", "Flow"]:
                raise ValueError(f"Invalid mode at row {row + 1}")
            if direction not in ["CW", "CCW"]:
                raise ValueError(f"Invalid direction at row {row + 1}")

            steps.append(Step(mode, value, duration_s, direction))

        return steps

    def run_sequence(self):
        try:
            if self.sequence_worker and self.sequence_worker.isRunning():
                QMessageBox.warning(self, "Sequence", "Sequence already running.")
                return

            pump = self.ensure_pump()
            steps = self.get_sequence_steps()
            if not steps:
                QMessageBox.information(self, "Sequence", "No sequence steps.")
                return

            self.sequence_worker = SequenceWorker(
                pump=pump,
                steps=steps,
                pump_head=self.get_pump_head(),
                tube_no=self.get_tube_no(),
            )
            self.sequence_worker.log.connect(self.log_msg)
            self.sequence_worker.finished_ok.connect(lambda: self.log_msg("Sequence finished"))
            self.sequence_worker.failed.connect(lambda e: QMessageBox.critical(self, "Sequence Error", e))
            self.sequence_worker.start()

        except Exception as e:
            QMessageBox.critical(self, "Sequence Error", str(e))

    def stop_sequence(self):
        if self.sequence_worker and self.sequence_worker.isRunning():
            self.sequence_worker.stop_requested()
            self.log_msg("Sequence stop requested")
        try:
            self.stop_pump()
        except Exception:
            pass

    def closeEvent(self, event):
        try:
            self.stop_monitor()
            self.stop_sequence()
            if self.pump:
                self.pump.stop()
                self.pump.close()
        except Exception:
            pass
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import *
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# =========================
# Pump Driver
# =========================
class BT1001L:
    def __init__(self, port="COM10", addr=1):
        self.port = port
        self.addr = addr
        self.ser = None

    def open(self):
        if not self.ser or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=8,
                parity='E',
                stopbits=1,
                timeout=1
            )

    def close(self):
        if self.ser:
            self.ser.close()

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([self._checksum(pdu)])

    def send(self, pdu):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(0.3)
        rx = self.ser.read_all()

        return frame, rx

    def set_speed(self, rpm):
        speed = int(rpm * 10)
        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([1, 1])
        return self.send(pdu)

    def stop(self):
        return self.set_speed(0)

    def read_speed(self):
        return self.send(b"DL")


# =========================
# Decode RPM
# =========================
def decode_rpm(rx):
    if not rx or rx[0] != 0xE9:
        return None

    speed = int.from_bytes(rx[5:7], "big")
    return speed / 10.0


# =========================
# Plot
# =========================
class PlotCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure()
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.t = []
        self.y = []

    def update_plot(self, t, rpm):
        self.t.append(t)
        self.y.append(rpm)

        if len(self.t) > 200:
            self.t = self.t[-200:]
            self.y = self.y[-200:]

        self.ax.clear()
        self.ax.plot(self.t, self.y)
        self.ax.set_ylabel("RPM")
        self.ax.set_xlabel("Time (s)")
        self.ax.grid()
        self.draw()


# =========================
# GUI
# =========================
class App(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("BT100 RPM Control")
        self.resize(800, 600)

        self.pump = None
        self.start_time = None

        # Widgets
        self.port = QComboBox()
        self.port.setEditable(True)

        self.rpm = QLineEdit("30")
        self.status = QLabel("Disconnected")
        self.feedback = QLabel("-")

        self.log = QTextEdit()
        self.log.setReadOnly(True)

        self.plot = PlotCanvas()

        # Buttons
        btn_connect = QPushButton("Connect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Monitor")

        # Layout
        layout = QVBoxLayout()

        grid = QGridLayout()
        grid.addWidget(QLabel("COM"), 0, 0)
        grid.addWidget(self.port, 0, 1)
        grid.addWidget(btn_connect, 0, 2)

        grid.addWidget(QLabel("RPM"), 1, 0)
        grid.addWidget(self.rpm, 1, 1)

        grid.addWidget(btn_start, 2, 0)
        grid.addWidget(btn_stop, 2, 1)
        grid.addWidget(btn_read, 2, 2)

        grid.addWidget(QLabel("Status"), 3, 0)
        grid.addWidget(self.status, 3, 1)

        grid.addWidget(QLabel("Feedback"), 4, 0)
        grid.addWidget(self.feedback, 4, 1)

        layout.addLayout(grid)
        layout.addWidget(btn_monitor)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)

        self.setLayout(layout)

        # Timer
        self.timer = QTimer()
        self.timer.timeout.connect(self.monitor)

        # Events
        btn_connect.clicked.connect(self.connect)
        btn_start.clicked.connect(self.start)
        btn_stop.clicked.connect(self.stop)
        btn_read.clicked.connect(self.read)
        btn_monitor.clicked.connect(self.start_monitor)

        self.refresh_ports()

    def refresh_ports(self):
        self.port.clear()
        ports = [p.device for p in serial.tools.list_ports.comports()]
        self.port.addItems(ports)
        for i in range(1, 50):
            self.port.addItem(f"COM{i}")

    def log_msg(self, txt):
        self.log.append(txt)

    def connect(self):
        self.pump = BT1001L(self.port.currentText(), 1)
        self.status.setText("Connected")

    def start(self):
        rpm = float(self.rpm.text())
        f, r = self.pump.set_speed(rpm)
        self.log_msg(f"TX {f.hex()}")
        self.status.setText(f"Running {rpm} RPM")

    def stop(self):
        self.pump.stop()
        self.status.setText("Stopped")

    def read(self):
        f, r = self.pump.read_speed()
        rpm = decode_rpm(r)
        self.feedback.setText(f"{rpm} RPM")
        self.log_msg(f"RX {r.hex()}")

    def start_monitor(self):
        self.start_time = time.time()
        self.timer.start(1000)

    def monitor(self):
        f, r = self.pump.read_speed()
        rpm = decode_rpm(r)

        if rpm is not None:
            t = time.time() - self.start_time
            self.plot.update_plot(t, rpm)
            self.feedback.setText(f"{rpm} RPM")

    def closeEvent(self, e):
        try:
            self.pump.stop()
            self.pump.close()
        except:
            pass


if __name__ == "__main__":
    app = QApplication(sys.argv)
    w = App()
    w.show()
    sys.exit(app.exec())

AttributeError: 'NoneType' object has no attribute 'set_speed'

AttributeError: 'NoneType' object has no attribute 'set_speed'

SerialException: could not open port 'COM4': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)

SystemExit: 0

In [1]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox, QLineEdit,
    QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout, QGroupBox,
    QMessageBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class BT1001L:
    def __init__(self, port="COM10", addr=1, timeout=1.0):
        self.port = port
        self.addr = int(addr)
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu: bytes) -> int:
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _frame(self, pdu: bytes) -> bytes:
        fcs = self._checksum(pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send(self, pdu: bytes, wait_s=0.25):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        rx = self.ser.read_all()
        return frame, rx

    def set_speed(self, rpm: float, start=True, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed_raw = int(round(rpm * 10))

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed_raw.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)

    def stop(self, cw=True):
        return self.set_speed(0.0, start=False, cw=cw)


def decode_rpm(rx: bytes):
    if not rx or len(rx) < 10:
        return None

    if rx[0] != 0xE9:
        return None

    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    speed_raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "rpm": speed_raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(5, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.rpm = []
        self.reset_axes()

    def reset_axes(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def add_point(self, t, rpm):
        self.t.append(t)
        self.rpm.append(rpm)

        if len(self.t) > 300:
            self.t = self.t[-300:]
            self.rpm = self.rpm[-300:]

        self.ax.clear()
        self.ax.plot(self.t, self.rpm)
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def clear(self):
        self.t.clear()
        self.rpm.clear()
        self.reset_axes()


class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr="1"):
        super().__init__(title)

        self.pump = None
        self.monitor_start = None

        self.port_box = QComboBox()
        self.port_box.setEditable(True)

        self.addr_edit = QLineEdit(default_addr)
        self.rpm_edit = QLineEdit("30.0")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.feedback = QLabel("-")
        self.status = QLabel("Disconnected")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(110)

        self.plot = PlotCanvas(title)

        self.monitor_timer = QTimer()
        self.monitor_timer.timeout.connect(self.monitor_tick)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh COM Ports")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Real-Time Read")
        btn_stop_monitor = QPushButton("Stop Real-Time Read")
        btn_clear = QPushButton("Clear Plot")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_clear.clicked.connect(self.plot.clear)

        grid.addWidget(QLabel("COM Port"), 0, 0)
        grid.addWidget(self.port_box, 0, 1)
        grid.addWidget(btn_refresh, 0, 2)

        grid.addWidget(QLabel("Address"), 1, 0)
        grid.addWidget(self.addr_edit, 1, 1)

        grid.addWidget(QLabel("Set RPM"), 2, 0)
        grid.addWidget(self.rpm_edit, 2, 1)

        grid.addWidget(QLabel("Direction"), 3, 0)
        grid.addWidget(self.dir_box, 3, 1)

        grid.addWidget(btn_connect, 4, 0)
        grid.addWidget(btn_disconnect, 4, 1)
        grid.addWidget(btn_start, 5, 0)
        grid.addWidget(btn_stop, 5, 1)
        grid.addWidget(btn_read, 5, 2)

        grid.addWidget(btn_monitor, 6, 0)
        grid.addWidget(btn_stop_monitor, 6, 1)
        grid.addWidget(btn_clear, 6, 2)

        grid.addWidget(QLabel("Feedback"), 7, 0)
        grid.addWidget(self.feedback, 7, 1, 1, 2)

        grid.addWidget(QLabel("Status"), 8, 0)
        grid.addWidget(self.status, 8, 1, 1, 2)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)
        self.setLayout(layout)

    def refresh_ports(self):
        current = self.port_box.currentText().strip()
        self.port_box.clear()

        ports = list(serial.tools.list_ports.comports())

        self.log.append("Available COM ports:")
        if ports:
            for p in ports:
                self.log.append(f"  {p.device} - {p.description}")
                self.port_box.addItem(p.device)
        else:
            self.log.append("  No COM ports detected")

        # Add extra COMxx options manually
        existing = {p.device.upper() for p in ports}
        for i in range(1, 257):
            name = f"COM{i}"
            if name not in existing:
                self.port_box.addItem(name)

        if current:
            self.port_box.setCurrentText(current)
        else:
            self.port_box.setCurrentText("COM10")

        self.log.append("")

    def get_port(self):
        return self.port_box.currentText().strip().upper()

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be 1 to 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be 0 to 100")
        return rpm

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()

        if self.pump is None:
            raise RuntimeError("Pump is not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()
        return self.pump

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)
            self.pump = BT1001L(self.get_port(), self.get_addr())
            self.pump.open()
            self.status.setText(f"Connected: {self.get_port()} addr={self.get_addr()}")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")
        except Exception as e:
            self.pump = None
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        try:
            self.stop_monitor()
            if self.pump:
                self.pump.close()
                self.pump = None
            self.status.setText("Disconnected")
            if show_log:
                self.log.append("Disconnected")
        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def start_pump(self):
        try:
            pump = self.ensure_pump()
            rpm = self.get_rpm()
            frame, rx = pump.set_speed(rpm, start=True, cw=self.get_cw())
            self.log_io(frame, rx)
            self.status.setText(f"Running {rpm} RPM {self.dir_box.currentText()}")
        except Exception as e:
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.stop(cw=self.get_cw())
            self.log_io(frame, rx)
            self.status.setText("Stopped")
        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            self.log_io(frame, rx)

            data = decode_rpm(rx)
            if data:
                text = f"{data['rpm']} RPM | {'running' if data['running'] else 'stopped'} | {data['direction']}"
                self.feedback.setText(text)
                self.status.setText(text)
            else:
                if self.get_addr() == 31:
                    self.feedback.setText("Broadcast address 31 cannot read RPM")
                else:
                    self.feedback.setText("No valid RPM response")
        except Exception as e:
            QMessageBox.critical(self, "Read Error", str(e))

    def start_monitor(self):
        try:
            self.ensure_pump()
            self.monitor_start = time.time()
            self.monitor_timer.start(1000)
            self.status.setText("Real-time RPM read started")
        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.monitor_timer.isActive():
            self.monitor_timer.stop()

    def monitor_tick(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            data = decode_rpm(rx)

            if data:
                t = time.time() - self.monitor_start
                rpm = data["rpm"]
                self.plot.add_point(t, rpm)

                text = f"{rpm} RPM | {'running' if data['running'] else 'stopped'} | {data['direction']}"
                self.feedback.setText(text)
                self.status.setText(text)
            else:
                self.feedback.setText("No valid RPM response")
        except Exception as e:
            self.stop_monitor()
            QMessageBox.critical(self, "Monitor Error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop(cw=self.get_cw())
        except Exception:
            pass


class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual BT100-1L RPM Controller")
        self.resize(1300, 760)

        self.pump1 = PumpPanel("Pump 1", default_addr="1")
        self.pump2 = PumpPanel("Pump 2", default_addr="2")

        btn_start_both = QPushButton("START BOTH")
        btn_stop_both = QPushButton("STOP BOTH")
        btn_read_both = QPushButton("READ BOTH RPM")
        btn_monitor_both = QPushButton("START REAL-TIME READ BOTH")
        btn_stop_monitor_both = QPushButton("STOP REAL-TIME READ BOTH")
        btn_emergency = QPushButton("EMERGENCY STOP")

        btn_emergency.setStyleSheet(
            "background-color: #b00020; color: white; font-weight: bold; padding: 8px;"
        )

        btn_start_both.clicked.connect(self.start_both)
        btn_stop_both.clicked.connect(self.stop_both)
        btn_read_both.clicked.connect(self.read_both)
        btn_monitor_both.clicked.connect(self.monitor_both)
        btn_stop_monitor_both.clicked.connect(self.stop_monitor_both)
        btn_emergency.clicked.connect(self.emergency_stop)

        top = QHBoxLayout()
        top.addWidget(self.pump1)
        top.addWidget(self.pump2)

        bottom = QHBoxLayout()
        bottom.addWidget(btn_start_both)
        bottom.addWidget(btn_stop_both)
        bottom.addWidget(btn_read_both)
        bottom.addWidget(btn_monitor_both)
        bottom.addWidget(btn_stop_monitor_both)
        bottom.addStretch()
        bottom.addWidget(btn_emergency)

        note = QLabel(
            "Use address 1 and 2 for independent readback. "
            "Address 31 broadcasts to all pumps but cannot return RPM feedback."
        )
        note.setWordWrap(True)

        layout = QVBoxLayout()
        layout.addLayout(top)
        layout.addLayout(bottom)
        layout.addWidget(note)

        self.setLayout(layout)

    def start_both(self):
        self.pump1.start_pump()
        self.pump2.start_pump()

    def stop_both(self):
        self.pump1.stop_pump()
        self.pump2.stop_pump()

    def read_both(self):
        self.pump1.read_rpm()
        self.pump2.read_rpm()

    def monitor_both(self):
        self.pump1.start_monitor()
        self.pump2.start_monitor()

    def stop_monitor_both(self):
        self.pump1.stop_monitor()
        self.pump2.stop_monitor()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        try:
            self.pump1.emergency_stop()
            self.pump2.emergency_stop()
            self.pump1.disconnect_pump(show_log=False)
            self.pump2.disconnect_pump(show_log=False)
        except Exception:
            pass
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())

SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
def refresh_ports(self):
    current = self.port_box.currentText().strip()
    self.port_box.clear()

    ports = list(serial.tools.list_ports.comports())

    self.log.append("Connected COM ports:")
    if ports:
        for p in ports:
            label = f"{p.device} - {p.description}"
            self.port_box.addItem(p.device)
            self.log.append(f"  {label}")
    else:
        self.log.append("  No COM ports detected")

    if current in [p.device for p in ports]:
        self.port_box.setCurrentText(current)
    elif ports:
        self.port_box.setCurrentText(ports[0].device)

    self.log.append("")

In [3]:
import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import *
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# =========================
# Pump Driver
# =========================
class BT1001L:
    def __init__(self, port="COM10", addr=1):
        self.port = port
        self.addr = addr
        self.ser = None

    def open(self):
        if not self.ser or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=8,
                parity='E',
                stopbits=1,
                timeout=1
            )

    def close(self):
        if self.ser:
            self.ser.close()

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([self._checksum(pdu)])

    def send(self, pdu):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(0.3)
        rx = self.ser.read_all()
        return frame, rx

    def set_speed(self, rpm, cw=True):
        speed = int(rpm * 10)
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([0x01, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + b"\x00\x00" + bytes([0x00, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL")


# =========================
# Decode RPM
# =========================
def decode_rpm(rx):
    if not rx or len(rx) < 10:
        return None
    if rx[0] != 0xE9:
        return None

    speed = int.from_bytes(rx[5:7], "big")
    running = bool(rx[7] & 0x01)
    direction = "CW" if (rx[8] & 0x01) else "CCW"

    return speed / 10.0, running, direction


# =========================
# Plot
# =========================
class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(4, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.y = []

    def update_plot(self, t, rpm):
        self.t.append(t)
        self.y.append(rpm)

        if len(self.t) > 200:
            self.t = self.t[-200:]
            self.y = self.y[-200:]

        self.ax.clear()
        self.ax.plot(self.t, self.y)
        self.ax.set_title(self.title)
        self.ax.set_ylabel("RPM")
        self.ax.set_xlabel("Time (s)")
        self.ax.grid()
        self.draw()


# =========================
# Pump Panel
# =========================
class PumpPanel(QGroupBox):
    def __init__(self, title, addr):
        super().__init__(title)

        self.addr = addr
        self.pump = None
        self.start_time = None

        self.port = QComboBox()
        self.port.setEditable(True)

        self.rpm = QLineEdit("30")

        self.dir = QComboBox()
        self.dir.addItems(["CW", "CCW"])

        self.feedback = QLabel("-")
        self.status = QLabel("Disconnected")

        self.plot = PlotCanvas(title)

        self.log = QTextEdit()
        self.log.setReadOnly(True)

        self.timer = QTimer()
        self.timer.timeout.connect(self.monitor)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh")
        btn_connect = QPushButton("Connect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Monitor")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect)
        btn_start.clicked.connect(self.start)
        btn_stop.clicked.connect(self.stop)
        btn_read.clicked.connect(self.read)
        btn_monitor.clicked.connect(self.start_monitor)

        grid.addWidget(QLabel("COM"), 0, 0)
        grid.addWidget(self.port, 0, 1)
        grid.addWidget(btn_refresh, 0, 2)

        grid.addWidget(QLabel("RPM"), 1, 0)
        grid.addWidget(self.rpm, 1, 1)

        grid.addWidget(QLabel("Direction"), 2, 0)
        grid.addWidget(self.dir, 2, 1)

        grid.addWidget(btn_connect, 3, 0)
        grid.addWidget(btn_start, 3, 1)
        grid.addWidget(btn_stop, 3, 2)

        grid.addWidget(btn_read, 4, 0)
        grid.addWidget(btn_monitor, 4, 1)

        grid.addWidget(QLabel("Status"), 5, 0)
        grid.addWidget(self.status, 5, 1)

        grid.addWidget(QLabel("Feedback"), 6, 0)
        grid.addWidget(self.feedback, 6, 1)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)

        self.setLayout(layout)

    def refresh_ports(self):
        self.port.clear()
        ports = list(serial.tools.list_ports.comports())

        if ports:
            for p in ports:
                self.port.addItem(p.device)
        else:
            self.log.append("No COM ports found")

    def connect(self):
        self.pump = BT1001L(self.port.currentText(), self.addr)
        self.status.setText("Connected")

    def start(self):
        rpm = float(self.rpm.text())
        cw = self.dir.currentText() == "CW"
        self.pump.set_speed(rpm, cw)
        self.status.setText(f"Running {rpm} RPM")

    def stop(self):
        cw = self.dir.currentText() == "CW"
        self.pump.stop(cw)
        self.status.setText("Stopped")

    def read(self):
        _, rx = self.pump.read_speed()
        data = decode_rpm(rx)
        if data:
            rpm, run, d = data
            self.feedback.setText(f"{rpm} RPM {d}")

    def start_monitor(self):
        self.start_time = time.time()
        self.timer.start(1000)

    def monitor(self):
        _, rx = self.pump.read_speed()
        data = decode_rpm(rx)

        if data:
            rpm, run, d = data
            t = time.time() - self.start_time
            self.plot.update_plot(t, rpm)
            self.feedback.setText(f"{rpm} RPM {d}")


# =========================
# Main Window
# =========================
class Main(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual Pump RPM Control")

        self.p1 = PumpPanel("Pump 1", 1)
        self.p2 = PumpPanel("Pump 2", 2)

        btn_start_all = QPushButton("START BOTH")
        btn_stop_all = QPushButton("STOP BOTH")

        btn_start_all.clicked.connect(self.start_all)
        btn_stop_all.clicked.connect(self.stop_all)

        layout = QVBoxLayout()
        top = QHBoxLayout()

        top.addWidget(self.p1)
        top.addWidget(self.p2)

        layout.addLayout(top)
        layout.addWidget(btn_start_all)
        layout.addWidget(btn_stop_all)

        self.setLayout(layout)

    def start_all(self):
        self.p1.start()
        self.p2.start()

    def stop_all(self):
        self.p1.stop()
        self.p2.stop()


# =========================
# Run
# =========================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    w = Main()
    w.show()
    sys.exit(app.exec())

SystemExit: 0

In [1]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class BT1001L:
    def __init__(self, port="COM10", addr=1):
        self.port = port
        self.addr = int(addr)
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([fcs])

    def send(self, pdu, wait_s=0.3):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        rx = self.ser.read_all()

        return frame, rx

    def set_speed(self, rpm, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx):
    if not rx or len(rx) < 10:
        return None

    if rx[0] != 0xE9:
        return None

    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    rpm_raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "rpm": rpm_raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


def get_real_com_ports():
    """
    Returns likely real connected USB/Moxa serial ports.
    Filters out many ghost/Bluetooth/virtual COM ports.
    """
    ports = list(serial.tools.list_ports.comports())
    valid = []

    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()

        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "UART" in desc
            or "SERIAL" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )

        is_ghost_or_bluetooth = (
            "BLUETOOTH" in desc
            or "BTHENUM" in hwid
            or "STANDARD SERIAL OVER BLUETOOTH" in desc
        )

        if is_usb_serial and not is_ghost_or_bluetooth:
            valid.append(p)

    return valid


class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(4, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.y = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def update_plot(self, t, rpm):
        self.t.append(t)
        self.y.append(rpm)

        if len(self.t) > 200:
            self.t = self.t[-200:]
            self.y = self.y[-200:]

        self.ax.clear()
        self.ax.plot(self.t, self.y)
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def clear(self):
        self.t.clear()
        self.y.clear()
        self.redraw()


class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)

        self.default_addr = default_addr
        self.pump = None
        self.start_time = None

        self.port_box = QComboBox()
        self.port_box.setEditable(False)

        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.feedback_label = QLabel("-")
        self.status_label = QLabel("Disconnected")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(120)

        self.plot = PlotCanvas(title)

        self.timer = QTimer()
        self.timer.timeout.connect(self.monitor_tick)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh COM Ports")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Real-Time RPM")
        btn_stop_monitor = QPushButton("Stop Real-Time RPM")
        btn_clear = QPushButton("Clear Plot")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_clear.clicked.connect(self.plot.clear)

        grid.addWidget(QLabel("COM Port"), 0, 0)
        grid.addWidget(self.port_box, 0, 1)
        grid.addWidget(btn_refresh, 0, 2)

        grid.addWidget(QLabel("Pump Address"), 1, 0)
        grid.addWidget(self.addr_edit, 1, 1)

        grid.addWidget(QLabel("Set RPM"), 2, 0)
        grid.addWidget(self.rpm_edit, 2, 1)

        grid.addWidget(QLabel("Direction"), 3, 0)
        grid.addWidget(self.dir_box, 3, 1)

        grid.addWidget(btn_connect, 4, 0)
        grid.addWidget(btn_disconnect, 4, 1)

        grid.addWidget(btn_start, 5, 0)
        grid.addWidget(btn_stop, 5, 1)
        grid.addWidget(btn_read, 5, 2)

        grid.addWidget(btn_monitor, 6, 0)
        grid.addWidget(btn_stop_monitor, 6, 1)
        grid.addWidget(btn_clear, 6, 2)

        grid.addWidget(QLabel("Feedback"), 7, 0)
        grid.addWidget(self.feedback_label, 7, 1, 1, 2)

        grid.addWidget(QLabel("Status"), 8, 0)
        grid.addWidget(self.status_label, 8, 1, 1, 2)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)

        self.setLayout(layout)

    def refresh_ports(self):
        current = self.port_box.currentText().strip()
        self.port_box.clear()

        ports = get_real_com_ports()

        self.log.append("Detected real COM ports:")

        if not ports:
            self.log.append("  No valid USB/Moxa COM ports found")
            self.log.append("")
            return

        for p in ports:
            label = f"{p.device} - {p.description}"
            self.port_box.addItem(p.device)
            self.log.append(f"  {label}")

        port_names = [p.device for p in ports]

        if current in port_names:
            self.port_box.setCurrentText(current)
        elif "COM10" in port_names:
            self.port_box.setCurrentText("COM10")
        else:
            self.port_box.setCurrentText(port_names[0])

        self.log.append("")

    def get_port(self):
        port = self.port_box.currentText().strip()
        if not port:
            raise ValueError("No COM port selected")
        return port

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be between 1 and 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be between 0 and 100")
        return rpm

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()

        if self.pump is None:
            raise RuntimeError("Pump is not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()

        return self.pump

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)

            self.pump = BT1001L(
                port=self.get_port(),
                addr=self.get_addr(),
            )
            self.pump.open()

            self.status_label.setText(f"Connected {self.get_port()} addr={self.get_addr()}")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")

        except Exception as e:
            self.pump = None
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        try:
            self.stop_monitor()

            if self.pump:
                self.pump.close()
                self.pump = None

            self.status_label.setText("Disconnected")

            if show_log:
                self.log.append("Disconnected")

        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def start_pump(self):
        try:
            pump = self.ensure_pump()
            rpm = self.get_rpm()
            cw = self.get_cw()

            frame, rx = pump.set_speed(rpm, cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText(f"Running {rpm} RPM {self.dir_box.currentText()}")

        except Exception as e:
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            cw = self.get_cw()

            frame, rx = pump.stop(cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText("Stopped")

        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            pump = self.ensure_pump()

            frame, rx = pump.read_speed()
            self.log_io(frame, rx)

            data = decode_rpm(rx)

            if data:
                text = (
                    f"{data['rpm']} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                if self.get_addr() == 31:
                    self.feedback_label.setText("Broadcast address 31 cannot read RPM")
                else:
                    self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            QMessageBox.critical(self, "Read Error", str(e))

    def start_monitor(self):
        try:
            self.ensure_pump()
            self.start_time = time.time()
            self.timer.start(1000)
            self.status_label.setText("Real-time RPM reading started")

        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.timer.isActive():
            self.timer.stop()

    def monitor_tick(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            data = decode_rpm(rx)

            if data:
                t = time.time() - self.start_time
                rpm = data["rpm"]

                self.plot.update_plot(t, rpm)

                text = (
                    f"{rpm} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            self.stop_monitor()
            QMessageBox.critical(self, "Monitor Error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop(cw=self.get_cw())
        except Exception:
            pass


class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual BT100-1L RPM Controller")
        self.resize(1350, 780)

        self.pump1 = PumpPanel("Pump 1", default_addr=1)
        self.pump2 = PumpPanel("Pump 2", default_addr=2)

        btn_refresh_all = QPushButton("REFRESH COM PORTS")
        btn_start_both = QPushButton("START BOTH")
        btn_stop_both = QPushButton("STOP BOTH")
        btn_read_both = QPushButton("READ BOTH RPM")
        btn_monitor_both = QPushButton("START REAL-TIME BOTH")
        btn_stop_monitor_both = QPushButton("STOP REAL-TIME BOTH")
        btn_emergency = QPushButton("EMERGENCY STOP")

        btn_emergency.setStyleSheet(
            "background-color: #b00020; color: white; font-weight: bold; padding: 8px;"
        )

        btn_refresh_all.clicked.connect(self.refresh_all_ports)
        btn_start_both.clicked.connect(self.start_both)
        btn_stop_both.clicked.connect(self.stop_both)
        btn_read_both.clicked.connect(self.read_both)
        btn_monitor_both.clicked.connect(self.monitor_both)
        btn_stop_monitor_both.clicked.connect(self.stop_monitor_both)
        btn_emergency.clicked.connect(self.emergency_stop)

        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        controls = QHBoxLayout()
        controls.addWidget(btn_refresh_all)
        controls.addWidget(btn_start_both)
        controls.addWidget(btn_stop_both)
        controls.addWidget(btn_read_both)
        controls.addWidget(btn_monitor_both)
        controls.addWidget(btn_stop_monitor_both)
        controls.addStretch()
        controls.addWidget(btn_emergency)

        note = QLabel(
            "Use address 1 and 2 for independent pump readback. "
            "Address 31 is broadcast: it can control pumps but cannot return RPM feedback."
        )
        note.setWordWrap(True)

        layout = QVBoxLayout()
        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)

        self.setLayout(layout)

    def refresh_all_ports(self):
        self.pump1.refresh_ports()
        self.pump2.refresh_ports()

    def start_both(self):
        self.pump1.start_pump()
        self.pump2.start_pump()

    def stop_both(self):
        self.pump1.stop_pump()
        self.pump2.stop_pump()

    def read_both(self):
        self.pump1.read_rpm()
        self.pump2.read_rpm()

    def monitor_both(self):
        self.pump1.start_monitor()
        self.pump2.start_monitor()

    def stop_monitor_both(self):
        self.pump1.stop_monitor()
        self.pump2.stop_monitor()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        try:
            self.pump1.emergency_stop()
            self.pump2.emergency_stop()
            self.pump1.disconnect_pump(show_log=False)
            self.pump2.disconnect_pump(show_log=False)
        except Exception:
            pass

        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class BT1001L:
    def __init__(self, port="COM4", addr=1):
        self.port = port
        self.addr = int(addr)
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([fcs])

    def send(self, pdu, wait_s=0.3):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        rx = self.ser.read_all()

        return frame, rx

    def set_speed(self, rpm, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx):
    if not rx or len(rx) < 10:
        return None

    if rx[0] != 0xE9:
        return None

    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    rpm_raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "rpm": rpm_raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


def get_real_com_ports():
    ports = list(serial.tools.list_ports.comports())
    valid = []

    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()

        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "UART" in desc
            or "SERIAL" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )

        is_ghost_or_bluetooth = (
            "BLUETOOTH" in desc
            or "BTHENUM" in hwid
            or "STANDARD SERIAL OVER BLUETOOTH" in desc
        )

        if is_usb_serial and not is_ghost_or_bluetooth:
            valid.append(p)

    return valid


class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(4, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.y = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def update_plot(self, t, rpm):
        self.t.append(t)
        self.y.append(rpm)

        if len(self.t) > 200:
            self.t = self.t[-200:]
            self.y = self.y[-200:]

        self.ax.clear()
        self.ax.plot(self.t, self.y)
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def clear(self):
        self.t.clear()
        self.y.clear()
        self.redraw()


class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)

        self.pump = None
        self.start_time = None

        self.port_box = QComboBox()
        self.port_box.setEditable(False)

        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.feedback_label = QLabel("-")
        self.status_label = QLabel("Disconnected")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(120)

        self.plot = PlotCanvas(title)

        self.timer = QTimer()
        self.timer.timeout.connect(self.monitor_tick)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh COM Ports")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Real-Time RPM")
        btn_stop_monitor = QPushButton("Stop Real-Time RPM")
        btn_clear = QPushButton("Clear Plot")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_clear.clicked.connect(self.plot.clear)

        grid.addWidget(QLabel("COM Port"), 0, 0)
        grid.addWidget(self.port_box, 0, 1)
        grid.addWidget(btn_refresh, 0, 2)

        grid.addWidget(QLabel("Pump Address"), 1, 0)
        grid.addWidget(self.addr_edit, 1, 1)

        grid.addWidget(QLabel("Set RPM"), 2, 0)
        grid.addWidget(self.rpm_edit, 2, 1)

        grid.addWidget(QLabel("Direction"), 3, 0)
        grid.addWidget(self.dir_box, 3, 1)

        grid.addWidget(btn_connect, 4, 0)
        grid.addWidget(btn_disconnect, 4, 1)

        grid.addWidget(btn_start, 5, 0)
        grid.addWidget(btn_stop, 5, 1)
        grid.addWidget(btn_read, 5, 2)

        grid.addWidget(btn_monitor, 6, 0)
        grid.addWidget(btn_stop_monitor, 6, 1)
        grid.addWidget(btn_clear, 6, 2)

        grid.addWidget(QLabel("Feedback"), 7, 0)
        grid.addWidget(self.feedback_label, 7, 1, 1, 2)

        grid.addWidget(QLabel("Status"), 8, 0)
        grid.addWidget(self.status_label, 8, 1, 1, 2)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)

        self.setLayout(layout)

    def refresh_ports(self):
        current_port = self.get_port(safe=True)

        self.port_box.clear()
        ports = get_real_com_ports()

        self.log.append("Detected real COM ports:")

        if not ports:
            self.log.append("  No valid USB/Moxa COM ports found\n")
            return

        for p in ports:
            display = f"{p.device} | {p.description}"
            self.port_box.addItem(display, p.device)
            self.log.append(f"  {display}")

        port_names = [p.device for p in ports]

        if current_port in port_names:
            index = port_names.index(current_port)
            self.port_box.setCurrentIndex(index)
        else:
            moxa_index = None
            for i, p in enumerate(ports):
                text = f"{p.description} {p.manufacturer or ''}".upper()
                if "MOXA" in text or "UPORT" in text:
                    moxa_index = i
                    break

            if moxa_index is not None:
                self.port_box.setCurrentIndex(moxa_index)
            else:
                self.port_box.setCurrentIndex(0)

        self.log.append("")

    def get_port(self, safe=False):
        port = self.port_box.currentData()

        if not port:
            text = self.port_box.currentText().strip()
            if "|" in text:
                port = text.split("|", 1)[0].strip()
            else:
                port = text

        if not port and not safe:
            raise ValueError("No COM port selected")

        return port

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be between 1 and 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be between 0 and 100")
        return rpm

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()

        if self.pump is None:
            raise RuntimeError("Pump is not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()
        return self.pump

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)

            self.pump = BT1001L(
                port=self.get_port(),
                addr=self.get_addr(),
            )
            self.pump.open()

            self.status_label.setText(f"Connected {self.get_port()} addr={self.get_addr()}")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")

        except Exception as e:
            self.pump = None
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        try:
            self.stop_monitor()

            if self.pump:
                self.pump.close()
                self.pump = None

            self.status_label.setText("Disconnected")

            if show_log:
                self.log.append("Disconnected")

        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def start_pump(self):
        try:
            pump = self.ensure_pump()
            rpm = self.get_rpm()
            cw = self.get_cw()

            frame, rx = pump.set_speed(rpm, cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText(f"Running {rpm} RPM {self.dir_box.currentText()}")

        except Exception as e:
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            cw = self.get_cw()

            frame, rx = pump.stop(cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText("Stopped")

        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            pump = self.ensure_pump()

            frame, rx = pump.read_speed()
            self.log_io(frame, rx)

            data = decode_rpm(rx)

            if data:
                text = (
                    f"{data['rpm']} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                if self.get_addr() == 31:
                    self.feedback_label.setText("Broadcast address 31 cannot read RPM")
                else:
                    self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            QMessageBox.critical(self, "Read Error", str(e))

    def start_monitor(self):
        try:
            self.ensure_pump()
            self.start_time = time.time()
            self.timer.start(1000)
            self.status_label.setText("Real-time RPM reading started")

        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.timer.isActive():
            self.timer.stop()

    def monitor_tick(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            data = decode_rpm(rx)

            if data:
                t = time.time() - self.start_time
                rpm = data["rpm"]

                self.plot.update_plot(t, rpm)

                text = (
                    f"{rpm} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            self.stop_monitor()
            QMessageBox.critical(self, "Monitor Error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop(cw=self.get_cw())
        except Exception:
            pass


class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual BT100-1L RPM Controller")
        self.resize(1350, 780)

        self.pump1 = PumpPanel("Pump 1", default_addr=1)
        self.pump2 = PumpPanel("Pump 2", default_addr=2)

        btn_refresh_all = QPushButton("REFRESH COM PORTS")
        btn_start_both = QPushButton("START BOTH")
        btn_stop_both = QPushButton("STOP BOTH")
        btn_read_both = QPushButton("READ BOTH RPM")
        btn_monitor_both = QPushButton("START REAL-TIME BOTH")
        btn_stop_monitor_both = QPushButton("STOP REAL-TIME BOTH")
        btn_emergency = QPushButton("EMERGENCY STOP")

        btn_emergency.setStyleSheet(
            "background-color: #b00020; color: white; font-weight: bold; padding: 8px;"
        )

        btn_refresh_all.clicked.connect(self.refresh_all_ports)
        btn_start_both.clicked.connect(self.start_both)
        btn_stop_both.clicked.connect(self.stop_both)
        btn_read_both.clicked.connect(self.read_both)
        btn_monitor_both.clicked.connect(self.monitor_both)
        btn_stop_monitor_both.clicked.connect(self.stop_monitor_both)
        btn_emergency.clicked.connect(self.emergency_stop)

        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        controls = QHBoxLayout()
        controls.addWidget(btn_refresh_all)
        controls.addWidget(btn_start_both)
        controls.addWidget(btn_stop_both)
        controls.addWidget(btn_read_both)
        controls.addWidget(btn_monitor_both)
        controls.addWidget(btn_stop_monitor_both)
        controls.addStretch()
        controls.addWidget(btn_emergency)

        note = QLabel(
            "Dropdown shows 'actual port | description'. "
            "Python opens the actual port before the vertical bar. "
            "Use address 1 and 2 for independent RPM feedback. "
            "Address 31 is broadcast: it can control pumps but cannot return RPM feedback."
        )
        note.setWordWrap(True)

        layout = QVBoxLayout()
        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)

        self.setLayout(layout)

    def refresh_all_ports(self):
        self.pump1.refresh_ports()
        self.pump2.refresh_ports()

    def start_both(self):
        self.pump1.start_pump()
        self.pump2.start_pump()

    def stop_both(self):
        self.pump1.stop_pump()
        self.pump2.stop_pump()

    def read_both(self):
        self.pump1.read_rpm()
        self.pump2.read_rpm()

    def monitor_both(self):
        self.pump1.start_monitor()
        self.pump2.start_monitor()

    def stop_monitor_both(self):
        self.pump1.stop_monitor()
        self.pump2.stop_monitor()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        try:
            self.pump1.emergency_stop()
            self.pump2.emergency_stop()
            self.pump1.disconnect_pump(show_log=False)
            self.pump2.disconnect_pump(show_log=False)
        except Exception:
            pass

        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

SystemExit: 0

In [1]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import re
import serial
import serial.tools.list_ports

from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# =========================
# COM PORT HELPERS
# =========================
def normalize_windows_com(port):
    """
    COM1-COM9 can be opened as COMx.
    COM10+ should be opened as \\.\COM10 on Windows.
    """
    port = port.strip().upper()
    if re.fullmatch(r"COM\d+", port):
        num = int(port[3:])
        if num >= 10:
            return r"\\.\{}".format(port)
    return port


def extract_display_com(port_info):
    """
    Example problem:
      p.device      = COM4
      p.description = MOXA USB Serial Port (COM10)

    We prefer the COM number inside parentheses when available.
    """
    text = f"{port_info.device} {port_info.description} {port_info.hwid}"
    matches = re.findall(r"COM\d+", text.upper())

    if matches:
        return matches[-1]  # usually COM10 from description

    return port_info.device


def get_real_com_ports():
    ports = list(serial.tools.list_ports.comports())
    valid = []

    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()

        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "UART" in desc
            or "SERIAL" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )

        is_bluetooth = (
            "BLUETOOTH" in desc
            or "BTHENUM" in hwid
            or "STANDARD SERIAL OVER BLUETOOTH" in desc
        )

        if is_usb_serial and not is_bluetooth:
            valid.append(p)

    return valid


# =========================
# PUMP DRIVER
# =========================
class BT1001L:
    def __init__(self, port="COM10", addr=1):
        self.port = port
        self.addr = int(addr)
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([fcs])

    def send(self, pdu, wait_s=0.3):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        rx = self.ser.read_all()

        return frame, rx

    def set_speed(self, rpm, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        speed = int(round(rpm * 10))

        state1 = 0x01
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx):
    if not rx or len(rx) < 10:
        return None

    if rx[0] != 0xE9:
        return None

    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    rpm_raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "rpm": rpm_raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


# =========================
# PLOT
# =========================
class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(4, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.y = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def update_plot(self, t, rpm):
        self.t.append(t)
        self.y.append(rpm)

        if len(self.t) > 200:
            self.t = self.t[-200:]
            self.y = self.y[-200:]

        self.ax.clear()
        self.ax.plot(self.t, self.y)
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def clear(self):
        self.t.clear()
        self.y.clear()
        self.redraw()


# =========================
# PUMP PANEL
# =========================
class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)

        self.pump = None
        self.start_time = None

        self.port_box = QComboBox()
        self.port_box.setEditable(False)

        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.feedback_label = QLabel("-")
        self.status_label = QLabel("Disconnected")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(120)

        self.plot = PlotCanvas(title)

        self.timer = QTimer()
        self.timer.timeout.connect(self.monitor_tick)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh COM Ports")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Real-Time RPM")
        btn_stop_monitor = QPushButton("Stop Real-Time RPM")
        btn_clear = QPushButton("Clear Plot")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_clear.clicked.connect(self.plot.clear)

        grid.addWidget(QLabel("COM Port"), 0, 0)
        grid.addWidget(self.port_box, 0, 1)
        grid.addWidget(btn_refresh, 0, 2)

        grid.addWidget(QLabel("Pump Address"), 1, 0)
        grid.addWidget(self.addr_edit, 1, 1)

        grid.addWidget(QLabel("Set RPM"), 2, 0)
        grid.addWidget(self.rpm_edit, 2, 1)

        grid.addWidget(QLabel("Direction"), 3, 0)
        grid.addWidget(self.dir_box, 3, 1)

        grid.addWidget(btn_connect, 4, 0)
        grid.addWidget(btn_disconnect, 4, 1)

        grid.addWidget(btn_start, 5, 0)
        grid.addWidget(btn_stop, 5, 1)
        grid.addWidget(btn_read, 5, 2)

        grid.addWidget(btn_monitor, 6, 0)
        grid.addWidget(btn_stop_monitor, 6, 1)
        grid.addWidget(btn_clear, 6, 2)

        grid.addWidget(QLabel("Feedback"), 7, 0)
        grid.addWidget(self.feedback_label, 7, 1, 1, 2)

        grid.addWidget(QLabel("Status"), 8, 0)
        grid.addWidget(self.status_label, 8, 1, 1, 2)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(self.log)

        self.setLayout(layout)

    def refresh_ports(self):
        current_port = self.get_port(safe=True)

        self.port_box.clear()
        ports = get_real_com_ports()

        self.log.append("Detected COM ports:")

        if not ports:
            self.log.append("  No valid USB/Moxa COM ports found\n")
            return

        real_ports = []

        for p in ports:
            real_com = extract_display_com(p)
            open_port = normalize_windows_com(real_com)

            display = f"{real_com} | device={p.device} | {p.description}"
            self.port_box.addItem(display, open_port)

            real_ports.append(open_port)
            self.log.append(f"  {display} -> opens {open_port}")

        if current_port in real_ports:
            self.port_box.setCurrentIndex(real_ports.index(current_port))
        else:
            moxa_index = None
            for i, p in enumerate(ports):
                text = f"{p.description} {p.manufacturer or ''}".upper()
                if "MOXA" in text or "UPORT" in text:
                    moxa_index = i
                    break

            if moxa_index is not None:
                self.port_box.setCurrentIndex(moxa_index)
            else:
                self.port_box.setCurrentIndex(0)

        self.log.append("")

    def get_port(self, safe=False):
        port = self.port_box.currentData()

        if not port:
            text = self.port_box.currentText().strip()
            if "|" in text:
                port = text.split("|", 1)[0].strip()
            else:
                port = text

            if port:
                port = normalize_windows_com(port)

        if not port and not safe:
            raise ValueError("No COM port selected")

        return port or ""

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be between 1 and 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be between 0 and 100")
        return rpm

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()

        if self.pump is None:
            raise RuntimeError("Pump is not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()

        return self.pump

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)

            selected_port = self.get_port()
            self.pump = BT1001L(
                port=selected_port,
                addr=self.get_addr(),
            )
            self.pump.open()

            self.status_label.setText(f"Connected {selected_port} addr={self.get_addr()}")
            self.log.append(f"Connected {selected_port} addr={self.get_addr()}")

        except Exception as e:
            self.pump = None
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        try:
            self.stop_monitor()

            if self.pump:
                self.pump.close()
                self.pump = None

            self.status_label.setText("Disconnected")

            if show_log:
                self.log.append("Disconnected")

        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def start_pump(self):
        try:
            pump = self.ensure_pump()
            rpm = self.get_rpm()
            cw = self.get_cw()

            frame, rx = pump.set_speed(rpm, cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText(f"Running {rpm} RPM {self.dir_box.currentText()}")

        except Exception as e:
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            cw = self.get_cw()

            frame, rx = pump.stop(cw=cw)
            self.log_io(frame, rx)

            self.status_label.setText("Stopped")

        except Exception as e:
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            pump = self.ensure_pump()

            frame, rx = pump.read_speed()
            self.log_io(frame, rx)

            data = decode_rpm(rx)

            if data:
                text = (
                    f"{data['rpm']} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                if self.get_addr() == 31:
                    self.feedback_label.setText("Broadcast address 31 cannot read RPM")
                else:
                    self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            QMessageBox.critical(self, "Read Error", str(e))

    def start_monitor(self):
        try:
            self.ensure_pump()
            self.start_time = time.time()
            self.timer.start(1000)
            self.status_label.setText("Real-time RPM reading started")

        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.timer.isActive():
            self.timer.stop()

    def monitor_tick(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            data = decode_rpm(rx)

            if data:
                t = time.time() - self.start_time
                rpm = data["rpm"]

                self.plot.update_plot(t, rpm)

                text = (
                    f"{rpm} RPM | "
                    f"{'running' if data['running'] else 'stopped'} | "
                    f"{data['direction']}"
                )
                self.feedback_label.setText(text)
                self.status_label.setText(text)
            else:
                self.feedback_label.setText("No valid RPM response")

        except Exception as e:
            self.stop_monitor()
            QMessageBox.critical(self, "Monitor Error", str(e))

    def emergency_stop(self):
        try:
            if self.pump:
                self.pump.stop(cw=self.get_cw())
        except Exception:
            pass


# =========================
# MAIN WINDOW
# =========================
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual BT100-1L RPM Controller")
        self.resize(1400, 800)

        self.pump1 = PumpPanel("Pump 1", default_addr=1)
        self.pump2 = PumpPanel("Pump 2", default_addr=2)

        btn_refresh_all = QPushButton("REFRESH COM PORTS")
        btn_start_both = QPushButton("START BOTH")
        btn_stop_both = QPushButton("STOP BOTH")
        btn_read_both = QPushButton("READ BOTH RPM")
        btn_monitor_both = QPushButton("START REAL-TIME BOTH")
        btn_stop_monitor_both = QPushButton("STOP REAL-TIME BOTH")
        btn_emergency = QPushButton("EMERGENCY STOP")

        btn_emergency.setStyleSheet(
            "background-color: #b00020; color: white; font-weight: bold; padding: 8px;"
        )

        btn_refresh_all.clicked.connect(self.refresh_all_ports)
        btn_start_both.clicked.connect(self.start_both)
        btn_stop_both.clicked.connect(self.stop_both)
        btn_read_both.clicked.connect(self.read_both)
        btn_monitor_both.clicked.connect(self.monitor_both)
        btn_stop_monitor_both.clicked.connect(self.stop_monitor_both)
        btn_emergency.clicked.connect(self.emergency_stop)

        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        controls = QHBoxLayout()
        controls.addWidget(btn_refresh_all)
        controls.addWidget(btn_start_both)
        controls.addWidget(btn_stop_both)
        controls.addWidget(btn_read_both)
        controls.addWidget(btn_monitor_both)
        controls.addWidget(btn_stop_monitor_both)
        controls.addStretch()
        controls.addWidget(btn_emergency)

        note = QLabel(
            "Dropdown format: REAL_COM | device=reported_device | description. "
            "For your Moxa example, select COM10 | device=COM4 | MOXA USB Serial Port (COM10). "
            "Address 31 is broadcast and cannot return RPM feedback."
        )
        note.setWordWrap(True)

        layout = QVBoxLayout()
        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)

        self.setLayout(layout)

    def refresh_all_ports(self):
        self.pump1.refresh_ports()
        self.pump2.refresh_ports()

    def start_both(self):
        self.pump1.start_pump()
        self.pump2.start_pump()

    def stop_both(self):
        self.pump1.stop_pump()
        self.pump2.stop_pump()

    def read_both(self):
        self.pump1.read_rpm()
        self.pump2.read_rpm()

    def monitor_both(self):
        self.pump1.start_monitor()
        self.pump2.start_monitor()

    def stop_monitor_both(self):
        self.pump1.stop_monitor()
        self.pump2.stop_monitor()

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        try:
            self.pump1.emergency_stop()
            self.pump2.emergency_stop()
            self.pump1.disconnect_pump(show_log=False)
            self.pump2.disconnect_pump(show_log=False)
        except Exception:
            pass

        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

<>:26: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<>:26: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_11588\3987426059.py:26: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  COM10+ should be opened as \\.\COM10 on Windows.


SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
C:\Users\User\AppData\Local\Temp\ipykernel_11588\3987426059.py:26: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  COM10+ should be opened as \\.\COM10 on Windows.


In [2]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import re
from dataclasses import dataclass

import serial
import serial.tools.list_ports

from PyQt6.QtCore import QThread, pyqtSignal, Qt
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox, QTableWidget, QTableWidgetItem
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# =========================
# COM HELPERS
# =========================
def normalize_windows_com(port):
    port = port.strip().upper()
    if re.fullmatch(r"COM\d+", port):
        if int(port[3:]) >= 10:
            return r"\\.\{}".format(port)
    return port


def extract_display_com(p):
    text = f"{p.device} {p.description} {p.hwid}"
    matches = re.findall(r"COM\d+", text.upper())
    return matches[-1] if matches else p.device


def get_real_com_ports():
    ports = list(serial.tools.list_ports.comports())
    valid = []

    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()

        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "SERIAL" in desc
            or "UART" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )

        is_bluetooth = "BLUETOOTH" in desc or "BTHENUM" in hwid

        if is_usb_serial and not is_bluetooth:
            valid.append(p)

    return valid


# =========================
# PUMP DRIVER
# =========================
class BT1001L:
    def __init__(self, port="COM10", addr=1):
        self.port = port
        self.addr = int(addr)
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([self._checksum(pdu)])

    def send(self, pdu, wait_s=0.25):
        self.open()
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        rx = self.ser.read_all()
        return frame, rx

    def set_speed(self, rpm, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        raw = int(round(rpm * 10))
        state1 = 0x01
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + raw.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx):
    if not rx or len(rx) < 10 or rx[0] != 0xE9:
        return None

    length = rx[2]
    pdu = rx[3:3 + length]

    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None

    raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]

    return {
        "rpm": raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


# =========================
# BACKGROUND WORKERS
# =========================
class MonitorWorker(QThread):
    data = pyqtSignal(float, float, bool, str, str)
    error = pyqtSignal(str)
    disconnected = pyqtSignal(str)

    def __init__(self, pump, interval_s=0.5):
        super().__init__()
        self.pump = pump
        self.interval_s = interval_s
        self.running = True
        self.t0 = time.time()

    def stop(self):
        self.running = False

    def run(self):
        while self.running:
            try:
                _, rx = self.pump.read_speed()
                decoded = decode_rpm(rx)

                if decoded is None:
                    self.error.emit("No valid RPM response")
                else:
                    t = time.time() - self.t0
                    self.data.emit(
                        t,
                        decoded["rpm"],
                        decoded["running"],
                        decoded["direction"],
                        decoded["raw"],
                    )

            except serial.SerialException as e:
                self.disconnected.emit(str(e))
                break
            except Exception as e:
                self.error.emit(str(e))

            time.sleep(self.interval_s)


@dataclass
class SequenceStep:
    rpm: float
    duration_s: float
    direction: str


class SequenceWorker(QThread):
    log = pyqtSignal(str)
    finished_ok = pyqtSignal()
    error = pyqtSignal(str)

    def __init__(self, pump, steps):
        super().__init__()
        self.pump = pump
        self.steps = steps
        self.running = True

    def stop(self):
        self.running = False

    def run(self):
        try:
            for i, step in enumerate(self.steps, 1):
                if not self.running:
                    break

                cw = step.direction == "CW"
                self.log.emit(f"Step {i}: {step.rpm} RPM, {step.duration_s}s, {step.direction}")

                self.pump.set_speed(step.rpm, cw=cw)

                t0 = time.time()
                while time.time() - t0 < step.duration_s:
                    if not self.running:
                        break
                    time.sleep(0.1)

                self.pump.stop(cw=cw)

            self.pump.stop()
            self.finished_ok.emit()

        except Exception as e:
            try:
                self.pump.stop()
            except Exception:
                pass
            self.error.emit(str(e))


# =========================
# PLOT
# =========================
class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(4, 3))
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)

        self.title = title
        self.t = []
        self.actual = []
        self.target = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def add_point(self, t, actual_rpm, target_rpm):
        self.t.append(t)
        self.actual.append(actual_rpm)
        self.target.append(target_rpm)

        if len(self.t) > 300:
            self.t = self.t[-300:]
            self.actual = self.actual[-300:]
            self.target = self.target[-300:]

        self.ax.clear()
        self.ax.plot(self.t, self.actual, label="Actual RPM")
        self.ax.plot(self.t, self.target, linestyle="--", label="Target RPM")
        self.ax.set_title(self.title)
        self.ax.set_xlabel("Time (s)")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.ax.legend()
        self.draw()

    def clear(self):
        self.t.clear()
        self.actual.clear()
        self.target.clear()
        self.redraw()


# =========================
# PUMP PANEL
# =========================
class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)

        self.pump = None
        self.monitor_worker = None
        self.sequence_worker = None
        self.target_rpm = 0.0

        self.port_box = QComboBox()
        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")
        self.interval_edit = QLineEdit("0.5")

        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.feedback_label = QLabel("-")
        self.status_label = QLabel("Disconnected")
        self.status_light = QLabel("●")
        self.status_light.setStyleSheet("color: gray; font-size: 22px;")

        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMinimumHeight(100)

        self.plot = PlotCanvas(title)

        self.sequence_table = QTableWidget(0, 3)
        self.sequence_table.setHorizontalHeaderLabels(["RPM", "Duration s", "Direction"])

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        layout = QVBoxLayout()
        grid = QGridLayout()

        btn_refresh = QPushButton("Refresh Ports")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read RPM")
        btn_monitor = QPushButton("Start Real-Time")
        btn_stop_monitor = QPushButton("Stop Real-Time")
        btn_toggle_dir = QPushButton("Toggle Direction")
        btn_clear = QPushButton("Clear Plot")

        btn_add_step = QPushButton("Add Step")
        btn_remove_step = QPushButton("Remove Step")
        btn_run_seq = QPushButton("Run Sequence")
        btn_stop_seq = QPushButton("Stop Sequence")

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_toggle_dir.clicked.connect(self.toggle_direction)
        btn_clear.clicked.connect(self.plot.clear)

        btn_add_step.clicked.connect(self.add_sequence_step)
        btn_remove_step.clicked.connect(self.remove_sequence_step)
        btn_run_seq.clicked.connect(self.run_sequence)
        btn_stop_seq.clicked.connect(self.stop_sequence)

        grid.addWidget(QLabel("Status"), 0, 0)
        grid.addWidget(self.status_light, 0, 1)
        grid.addWidget(self.status_label, 0, 2)

        grid.addWidget(QLabel("COM Port"), 1, 0)
        grid.addWidget(self.port_box, 1, 1)
        grid.addWidget(btn_refresh, 1, 2)

        grid.addWidget(QLabel("Address"), 2, 0)
        grid.addWidget(self.addr_edit, 2, 1)

        grid.addWidget(QLabel("Target RPM"), 3, 0)
        grid.addWidget(self.rpm_edit, 3, 1)

        grid.addWidget(QLabel("Direction"), 4, 0)
        grid.addWidget(self.dir_box, 4, 1)
        grid.addWidget(btn_toggle_dir, 4, 2)

        grid.addWidget(QLabel("Poll interval s"), 5, 0)
        grid.addWidget(self.interval_edit, 5, 1)

        grid.addWidget(btn_connect, 6, 0)
        grid.addWidget(btn_disconnect, 6, 1)
        grid.addWidget(btn_start, 7, 0)
        grid.addWidget(btn_stop, 7, 1)
        grid.addWidget(btn_read, 7, 2)

        grid.addWidget(btn_monitor, 8, 0)
        grid.addWidget(btn_stop_monitor, 8, 1)
        grid.addWidget(btn_clear, 8, 2)

        grid.addWidget(QLabel("Feedback"), 9, 0)
        grid.addWidget(self.feedback_label, 9, 1, 1, 2)

        seq_buttons = QHBoxLayout()
        seq_buttons.addWidget(btn_add_step)
        seq_buttons.addWidget(btn_remove_step)
        seq_buttons.addWidget(btn_run_seq)
        seq_buttons.addWidget(btn_stop_seq)

        layout.addLayout(grid)
        layout.addWidget(self.plot)
        layout.addWidget(QLabel("Sequence"))
        layout.addWidget(self.sequence_table)
        layout.addLayout(seq_buttons)
        layout.addWidget(self.log)

        self.setLayout(layout)

    def set_status(self, text, color):
        self.status_label.setText(text)
        self.status_light.setStyleSheet(f"color: {color}; font-size: 22px;")

    def refresh_ports(self):
        current = self.get_port(safe=True)
        self.port_box.clear()

        ports = get_real_com_ports()
        self.log.append("Detected COM ports:")

        for p in ports:
            real_com = extract_display_com(p)
            open_port = normalize_windows_com(real_com)
            display = f"{real_com} | device={p.device} | {p.description}"
            self.port_box.addItem(display, open_port)
            self.log.append(f"  {display} -> opens {open_port}")

        if not ports:
            self.log.append("  No valid ports found")
        else:
            open_ports = [self.port_box.itemData(i) for i in range(self.port_box.count())]
            if current in open_ports:
                self.port_box.setCurrentIndex(open_ports.index(current))
            else:
                self.port_box.setCurrentIndex(0)

        self.log.append("")

    def get_port(self, safe=False):
        port = self.port_box.currentData()
        if not port and not safe:
            raise ValueError("No COM port selected")
        return port or ""

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be 1 to 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be 0 to 100")
        return rpm

    def get_interval(self):
        return max(0.2, float(self.interval_edit.text().strip()))

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def connect_pump(self):
        try:
            self.disconnect_pump(show_log=False)
            self.pump = BT1001L(self.get_port(), self.get_addr())
            self.pump.open()
            self.set_status(f"Connected {self.get_port()} addr={self.get_addr()}", "green")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")
        except Exception as e:
            self.pump = None
            self.set_status("Connection failed", "red")
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_log=True):
        self.stop_monitor()
        self.stop_sequence()

        try:
            if self.pump:
                try:
                    self.pump.stop(cw=self.get_cw())  # feature 17: auto stop on disconnect
                except Exception:
                    pass
                self.pump.close()
                self.pump = None

            self.set_status("Disconnected", "gray")
            if show_log:
                self.log.append("Disconnected")

        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def ensure_pump(self):
        if self.pump is None:
            self.connect_pump()
        if self.pump is None:
            raise RuntimeError("Pump not connected")

        self.pump.port = self.get_port()
        self.pump.addr = self.get_addr()
        return self.pump

    def log_io(self, frame, rx):
        self.log.append(f"TX: {frame.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}")
        self.log.append("")

    def start_pump(self):
        try:
            pump = self.ensure_pump()
            rpm = self.get_rpm()
            self.target_rpm = rpm

            frame, rx = pump.set_speed(rpm, cw=self.get_cw())
            self.log_io(frame, rx)

            self.set_status(f"Running {rpm} RPM {self.dir_box.currentText()}", "green")

        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            pump = self.ensure_pump()
            self.target_rpm = 0.0

            frame, rx = pump.stop(cw=self.get_cw())
            self.log_io(frame, rx)

            self.set_status("Stopped", "orange")

        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            pump = self.ensure_pump()
            frame, rx = pump.read_speed()
            self.log_io(frame, rx)

            data = decode_rpm(rx)
            if data:
                self.update_feedback(data["rpm"], data["running"], data["direction"])
            else:
                self.feedback_label.setText("No valid RPM response")
                self.set_status("No response", "orange")

        except Exception as e:
            self.set_status("Disconnected/Error", "red")
            QMessageBox.critical(self, "Read Error", str(e))

    def update_feedback(self, actual_rpm, running, direction):
        error = actual_rpm - self.target_rpm
        text = (
            f"Target: {self.target_rpm:.1f} RPM | "
            f"Actual: {actual_rpm:.1f} RPM | "
            f"Error: {error:+.1f} RPM | "
            f"{'running' if running else 'stopped'} | {direction}"
        )
        self.feedback_label.setText(text)
        self.set_status(text, "green" if running else "orange")

    def start_monitor(self):
        try:
            pump = self.ensure_pump()
            self.stop_monitor()

            self.monitor_worker = MonitorWorker(pump, interval_s=self.get_interval())
            self.monitor_worker.data.connect(self.handle_monitor_data)
            self.monitor_worker.error.connect(self.handle_monitor_error)
            self.monitor_worker.disconnected.connect(self.handle_disconnect_detected)
            self.monitor_worker.start()

            self.set_status("Real-time polling started", "green")

        except Exception as e:
            self.set_status("Monitor error", "red")
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.monitor_worker and self.monitor_worker.isRunning():
            self.monitor_worker.stop()
            self.monitor_worker.wait(1500)
        self.monitor_worker = None

    def handle_monitor_data(self, t, actual_rpm, running, direction, raw):
        self.plot.add_point(t, actual_rpm, self.target_rpm)
        self.update_feedback(actual_rpm, running, direction)

    def handle_monitor_error(self, msg):
        self.feedback_label.setText(msg)
        self.set_status("Polling warning", "orange")

    def handle_disconnect_detected(self, msg):
        self.set_status("Disconnected detected", "red")
        self.feedback_label.setText("COM disconnected; attempting safe stop/close")
        try:
            if self.pump:
                self.pump.close()
        except Exception:
            pass
        self.pump = None

    def toggle_direction(self):
        self.dir_box.setCurrentText("CCW" if self.dir_box.currentText() == "CW" else "CW")

    def add_sequence_step(self):
        row = self.sequence_table.rowCount()
        self.sequence_table.insertRow(row)
        values = [self.rpm_edit.text().strip(), "5", self.dir_box.currentText()]

        for col, value in enumerate(values):
            self.sequence_table.setItem(row, col, QTableWidgetItem(value))

    def remove_sequence_step(self):
        row = self.sequence_table.currentRow()
        if row >= 0:
            self.sequence_table.removeRow(row)

    def get_steps(self):
        steps = []
        for row in range(self.sequence_table.rowCount()):
            rpm = float(self.sequence_table.item(row, 0).text())
            duration = float(self.sequence_table.item(row, 1).text())
            direction = self.sequence_table.item(row, 2).text().strip().upper()

            if direction not in ["CW", "CCW"]:
                raise ValueError(f"Invalid direction at row {row + 1}")

            steps.append(SequenceStep(rpm, duration, direction))

        return steps

    def run_sequence(self):
        try:
            if self.sequence_worker and self.sequence_worker.isRunning():
                QMessageBox.warning(self, "Sequence", "Sequence already running.")
                return

            pump = self.ensure_pump()
            steps = self.get_steps()

            if not steps:
                QMessageBox.information(self, "Sequence", "No steps added.")
                return

            self.sequence_worker = SequenceWorker(pump, steps)
            self.sequence_worker.log.connect(self.log.append)
            self.sequence_worker.finished_ok.connect(lambda: self.set_status("Sequence finished", "orange"))
            self.sequence_worker.error.connect(lambda e: QMessageBox.critical(self, "Sequence Error", e))
            self.sequence_worker.start()

            self.set_status("Sequence running", "green")

        except Exception as e:
            QMessageBox.critical(self, "Sequence Error", str(e))

    def stop_sequence(self):
        if self.sequence_worker and self.sequence_worker.isRunning():
            self.sequence_worker.stop()
            self.sequence_worker.wait(1500)
        self.sequence_worker = None

    def emergency_stop(self):
        try:
            self.stop_monitor()
            self.stop_sequence()
            if self.pump:
                self.pump.stop(cw=self.get_cw())
            self.target_rpm = 0.0
            self.set_status("Emergency stopped", "red")
        except Exception:
            pass


# =========================
# MAIN WINDOW
# =========================
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Dual BT100-1L RPM Controller")
        self.resize(1500, 880)

        self.pump1 = PumpPanel("Pump 1", 1)
        self.pump2 = PumpPanel("Pump 2", 2)

        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        btn_refresh = QPushButton("REFRESH PORTS")
        btn_start_both = QPushButton("START BOTH")
        btn_stop_both = QPushButton("STOP BOTH")
        btn_monitor_both = QPushButton("START REAL-TIME BOTH")
        btn_stop_monitor_both = QPushButton("STOP REAL-TIME BOTH")
        btn_emergency = QPushButton("EMERGENCY STOP")

        btn_emergency.setStyleSheet("background-color: #b00020; color: white; font-weight: bold; padding: 8px;")

        btn_refresh.clicked.connect(lambda: [self.pump1.refresh_ports(), self.pump2.refresh_ports()])
        btn_start_both.clicked.connect(lambda: [self.pump1.start_pump(), self.pump2.start_pump()])
        btn_stop_both.clicked.connect(lambda: [self.pump1.stop_pump(), self.pump2.stop_pump()])
        btn_monitor_both.clicked.connect(lambda: [self.pump1.start_monitor(), self.pump2.start_monitor()])
        btn_stop_monitor_both.clicked.connect(lambda: [self.pump1.stop_monitor(), self.pump2.stop_monitor()])
        btn_emergency.clicked.connect(self.emergency_stop)

        controls = QHBoxLayout()
        controls.addWidget(btn_refresh)
        controls.addWidget(btn_start_both)
        controls.addWidget(btn_stop_both)
        controls.addWidget(btn_monitor_both)
        controls.addWidget(btn_stop_monitor_both)
        controls.addStretch()
        controls.addWidget(btn_emergency)

        note = QLabel(
            "Added: target vs actual RPM, sequence mode, direction toggle, color status, "
            "threaded real-time polling, and auto-stop on disconnect."
        )
        note.setWordWrap(True)

        layout = QVBoxLayout()
        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)
        self.setLayout(layout)

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        self.pump1.disconnect_pump(show_log=False)
        self.pump2.disconnect_pump(show_log=False)
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

SystemExit: 0